# UC Governance Explorer — Tags, Comments & Lineage
This notebook demonstrates how to query Unity Catalog **system tables** (`system.information_schema`, `system.access`) to:
- List column tags and comments across any catalog/schema
- Filter for columns tagged as **PII** or **critical** (configurable tag names)
- Trace upstream/downstream lineage for Gold tables
- Build a **"What breaks if we change X?"** impact analysis report

### Parameters
| Widget | Purpose | Current value |
|---|---|---|
| `catalog_name` | Target catalog | `asvi_catalog` |
| `schema_name` | Target schema | `iceberg_schema` |
| `pii_tag` | PII tag key to search for | `pii` |
| `sensitivity_tag` | Sensitivity tag key | `asvi_sensitivity` |
| `critical_tag` | Business-critical tag key | `asvi_critical` |

## 1. Setup: Tag Columns as PII and Critical
Before querying tags, let's apply some governance tags to our pipeline tables. This demonstrates how tags would be set in practice.

In [0]:
%sql
-- Tag PII columns across the pipeline
-- Note: SET TAGS syntax requires literal tag names; :pii_tag/:sensitivity_tag/:critical_tag are used in query cells
ALTER TABLE IDENTIFIER(:catalog_name || '.' || :schema_name || '.silver_orders_enriched')
  ALTER COLUMN customer_name SET TAGS (:pii_tag = 'true', :sensitivity_tag = 'High');

ALTER TABLE IDENTIFIER(:catalog_name || '.' || :schema_name || '.silver_orders_enriched')
  ALTER COLUMN customer_email SET TAGS (:pii_tag  = 'true', :sensitivity_tag = 'Critical');

ALTER TABLE IDENTIFIER(:catalog_name || '.' || :schema_name || '.silver_reviews_enriched')
  ALTER COLUMN customer_name SET TAGS (:pii_tag  = 'true', :sensitivity_tag = 'High');

ALTER TABLE IDENTIFIER(:catalog_name || '.' || :schema_name || '.silver_reviews_enriched')
  ALTER COLUMN customer_email SET TAGS (:pii_tag  = 'true', :sensitivity_tag = 'Critical');

-- Tag business-critical columns on Gold tables
ALTER TABLE IDENTIFIER(:catalog_name || '.' || :schema_name || '.gold_sales_summary')
  ALTER COLUMN total_revenue SET TAGS (:critical_tag = 'true', 'domain' = 'finance');

ALTER TABLE IDENTIFIER(:catalog_name || '.' || :schema_name || '.gold_sales_summary')
  ALTER COLUMN total_orders SET TAGS (:critical_tag = 'true', 'domain' = 'finance');

ALTER TABLE IDENTIFIER(:catalog_name || '.' || :schema_name || '.gold_conversion_funnel')
  ALTER COLUMN conversion_rate_pct SET TAGS (:critical_tag = 'true', 'domain' = 'sales');

In [0]:
%sql
SHOW TABLES IN IDENTIFIER(:catalog_name || '.' || :schema_name);

In [0]:
%sql
DESCRIBE IDENTIFIER(:catalog_name || '.' || :schema_name || '.gold_sales_summary')

## 2. List Column Tags and Comments
Query `information_schema.column_tags` and `information_schema.columns` to inventory all tagged and documented columns in the Iceberg pipeline schema.

In [0]:
%sql
-- All column tags in the target schema
SELECT
    table_name,
    column_name,
    tag_name,
    tag_value
FROM system.information_schema.column_tags
WHERE catalog_name = :catalog_name
  AND schema_name = :schema_name
ORDER BY table_name, column_name, tag_name

In [0]:
%sql
-- All columns with their comments (documentation completeness check)
SELECT
    table_name,
    column_name,
    data_type,
    comment,
    CASE WHEN comment IS NULL THEN '❌ Missing' ELSE '✅ Documented' END AS doc_status
FROM system.information_schema.columns
WHERE table_catalog = :catalog_name
  AND table_schema = :schema_name
ORDER BY table_name, ordinal_position

## 3. Filter for PII and Critical Columns
Identify all columns tagged as PII or marked as business-critical — essential for compliance audits and access reviews.

In [0]:
%sql
-- Find all PII-tagged columns
SELECT
    table_name,
    column_name,
    tag_name,
    tag_value
FROM system.information_schema.column_tags
WHERE catalog_name = :catalog_name
  AND schema_name = :schema_name
  AND (
    tag_name = :pii_tag
    OR (tag_name = :sensitivity_tag AND tag_value IN ('High', 'Critical'))
  )
ORDER BY table_name, column_name

In [0]:
%sql
-- Find all business-critical columns
SELECT
    table_name,
    column_name,
    tag_name,
    tag_value
FROM system.information_schema.column_tags
WHERE catalog_name = :catalog_name
  AND schema_name = :schema_name
  AND tag_name = :critical_tag
  AND tag_value = 'true'
ORDER BY table_name, column_name

In [0]:
%sql
-- Combined governance report: PII and Critical columns
SELECT
    ct.table_name,
    ct.column_name,
    c.data_type,
    COLLECT_SET(CONCAT(ct.tag_name, '=', ct.tag_value)) AS tags,
    c.comment
FROM system.information_schema.column_tags ct
JOIN system.information_schema.columns c
  ON ct.catalog_name = c.table_catalog
  AND ct.schema_name = c.table_schema
  AND ct.table_name = c.table_name
  AND ct.column_name = c.column_name
WHERE ct.catalog_name = :catalog_name
  AND ct.schema_name = :schema_name
  AND ct.tag_name IN (:pii_tag, :critical_tag)
GROUP BY ct.table_name, ct.column_name, c.data_type, c.comment
ORDER BY ct.table_name, ct.column_name

## 4. Upstream / Downstream Lineage for a Gold Table
Use `system.access.table_lineage` and `system.access.column_lineage` to trace what feeds into `gold_sales_summary` and what depends on it.

In [0]:
%sql
-- What tables feed INTO gold_sales_summary? (upstream)
SELECT DISTINCT
    source_table_full_name AS upstream_table,
    source_type,
    MAX(event_time) AS last_seen
FROM system.access.table_lineage
WHERE target_table_full_name = :catalog_name || '.' || :schema_name || '.gold_sales_summary'
GROUP BY source_table_full_name, source_type
ORDER BY last_seen DESC

In [0]:
%sql
-- What tables CONSUME gold_sales_summary? (downstream)
SELECT DISTINCT
    target_table_full_name AS downstream_table,
    target_type,
    MAX(event_time) AS last_seen
FROM system.access.table_lineage
WHERE source_table_full_name = :catalog_name || '.' || :schema_name || '.gold_sales_summary'
GROUP BY target_table_full_name, target_type
ORDER BY last_seen DESC

In [0]:
%sql
-- Recursive upstream lineage: trace the full chain back to Bronze
WITH RECURSIVE lineage_chain AS (
    -- Anchor: direct upstream of our Gold table
    SELECT
        source_table_full_name,
        target_table_full_name,
        1 AS depth,
        ARRAY(target_table_full_name) AS path
    FROM system.access.table_lineage
    WHERE target_table_full_name = :catalog_name || '.' || :schema_name || '.gold_sales_summary'

    UNION ALL

    -- Recurse: follow each source upstream
    SELECT
        t.source_table_full_name,
        t.target_table_full_name,
        l.depth + 1,
        ARRAY_APPEND(l.path, t.target_table_full_name)
    FROM system.access.table_lineage t
    JOIN lineage_chain l
      ON t.target_table_full_name = l.source_table_full_name
    WHERE l.depth < 5
      AND NOT ARRAY_CONTAINS(l.path, t.source_table_full_name)  -- prevent cycles
)
SELECT DISTINCT
    source_table_full_name,
    target_table_full_name,
    depth AS hops_from_gold
FROM lineage_chain
ORDER BY hops_from_gold, source_table_full_name

In [0]:
%sql
-- Column-level lineage: where does total_revenue come from?
SELECT
    source_table_full_name,
    source_column_name,
    target_table_full_name,
    target_column_name,
    event_time
FROM system.access.column_lineage
WHERE target_table_full_name = :catalog_name || '.' || :schema_name || '.gold_sales_summary'
  AND target_column_name IN ('total_revenue', 'total_orders', 'avg_order_value')
ORDER BY target_column_name, event_time DESC

## 5. "What Breaks If We Change X?" — Impact Analysis Report
Combine lineage + tags to assess the blast radius of modifying a source column. Example: what happens if we change `customer_email` in `silver_orders_enriched`?

In [0]:
%sql
-- IMPACT ANALYSIS: What breaks if we change `customer_email` in silver_orders_enriched?
-- Find all downstream columns that depend on this column
SELECT
    'COLUMN DEPENDENCY' AS analysis_type,
    target_table_full_name AS affected_table,
    target_column_name AS affected_column,
    event_time AS last_observed
FROM system.access.column_lineage
WHERE source_table_full_name = :catalog_name || '.' || :schema_name || '.silver_orders_enriched'
  AND source_column_name = 'customer_email'
ORDER BY target_table_full_name, target_column_name

In [0]:
%sql
-- FULL IMPACT REPORT: Combine downstream lineage with governance tags
-- Shows affected downstream columns AND whether they carry PII/critical tags
WITH downstream_cols AS (
    SELECT DISTINCT
        target_table_full_name,
        target_column_name
    FROM system.access.column_lineage
    WHERE source_table_full_name = :catalog_name || '.' || :schema_name || '.silver_orders_enriched'
      AND source_column_name = 'customer_email'
),
downstream_tables AS (
    SELECT DISTINCT
        target_table_full_name
    FROM system.access.table_lineage
    WHERE source_table_full_name = :catalog_name || '.' || :schema_name || '.silver_orders_enriched'
),
affected_tags AS (
    SELECT
        CONCAT(ct.catalog_name, '.', ct.schema_name, '.', ct.table_name) AS full_table_name,
        ct.column_name,
        COLLECT_SET(CONCAT(ct.tag_name, '=', ct.tag_value)) AS governance_tags
    FROM system.information_schema.column_tags ct
    WHERE ct.catalog_name = :catalog_name
      AND ct.schema_name = :schema_name
      AND CONCAT(ct.catalog_name, '.', ct.schema_name, '.', ct.table_name) IN (
        SELECT target_table_full_name FROM downstream_tables
    )
    GROUP BY ct.catalog_name, ct.schema_name, ct.table_name, ct.column_name
)
SELECT
    '⚠️ IMPACT REPORT' AS report,
    'Source: silver_orders_enriched.customer_email' AS change_target,
    dt.target_table_full_name AS affected_table,
    dc.target_column_name AS affected_column,
    COALESCE(at.governance_tags, ARRAY()) AS governance_tags,
    CASE
        WHEN ARRAY_CONTAINS(COALESCE(at.governance_tags, ARRAY()), :pii_tag || '=true') THEN '🔴 HIGH - PII propagation risk'
        WHEN ARRAY_CONTAINS(COALESCE(at.governance_tags, ARRAY()), :critical_tag || '=true') THEN '🟡 MEDIUM - Business critical'
        ELSE '🟢 LOW'
    END AS risk_level
FROM downstream_tables dt
LEFT JOIN downstream_cols dc
  ON dt.target_table_full_name = dc.target_table_full_name
LEFT JOIN affected_tags at
  ON dt.target_table_full_name = at.full_table_name
  AND dc.target_column_name = at.column_name
ORDER BY risk_level, affected_table

## 6. Governance Summary — Documentation Gaps
Quick audit to find tables and columns in the pipeline that are missing comments or tags.

In [0]:
%sql
-- Governance gap analysis: columns without any tags in the pipeline
SELECT
    c.table_name,
    c.column_name,
    c.data_type,
    c.comment,
    CASE WHEN ct.column_name IS NOT NULL THEN '✅ Tagged' ELSE '❌ No Tags' END AS tag_status,
    CASE WHEN c.comment IS NOT NULL THEN '✅ Documented' ELSE '❌ No Comment' END AS doc_status
FROM system.information_schema.columns c
LEFT JOIN (
    SELECT DISTINCT schema_name, table_name, column_name
    FROM system.information_schema.column_tags
    WHERE catalog_name = :catalog_name
      AND schema_name = :schema_name
) ct
  ON c.table_schema = ct.schema_name
  AND c.table_name = ct.table_name
  AND c.column_name = ct.column_name
WHERE c.table_catalog = :catalog_name
  AND c.table_schema = :schema_name
ORDER BY
    CASE WHEN ct.column_name IS NULL AND c.comment IS NULL THEN 0 ELSE 1 END,
    c.table_name, c.column_name

## Key Takeaways
- **`system.information_schema.column_tags`** — query tags applied to columns across any catalog (PII, critical, domain, etc.)
- **`system.information_schema.columns`** — check column comments for documentation completeness
- **`system.access.table_lineage`** — trace upstream/downstream table dependencies
- **`system.access.column_lineage`** — trace column-level data flow for precise impact analysis
- **Combine lineage + tags** to build automated "blast radius" reports before making schema changes
- **Parameterized** — change `catalog_name`, `schema_name`, and tag name widgets to explore governance across any pipeline

> **Tip:** Schedule these queries as a Databricks Job to generate weekly governance reports, or surface them in an AI/BI Dashboard for continuous monitoring.